Objective: Train a classification model to predict late_delivery_risk. We'll use a Random Forest model because it's powerful and provides insights into which factors are most important.

In [1]:
import pandas as pd
import numpy as np
import joblib
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, accuracy_score

In [2]:
# Load the feature-engineered data
df = pd.read_csv('../data/processed/final_features.csv')

In [3]:
# Define the target variable
target = 'late_delivery_risk'
y = df[target]

# Select features for the model
# NOTE: We do NOT one-hot encode here. The pipeline will do it.
features = [
    'days_for_shipment_scheduled', 'benefit_per_order', 'sales_per_customer',
    'category_name', 'customer_segment', 'market', 'order_region',
    'shipping_mode', 'order_month', 'order_weekday',

    # --- ADD NEW FEATURES ---
    'is_weekend',
    'profit_per_day_scheduled',
    'category_late_rate',
    'customer_late_rate'
]
X = df[features]

In [4]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Training data shape: (144415, 14)
Testing data shape: (36104, 14)


In [5]:
# Identify categorical and numerical features
categorical_features = ['category_name', 'customer_segment', 'market', 'order_region', 'shipping_mode']
numerical_features = [
    'days_for_shipment_scheduled', 'benefit_per_order', 'sales_per_customer', 
    'order_month', 'order_weekday',

    # --- ADD NEW FEATURES ---
    'is_weekend',
    'profit_per_day_scheduled',
    'category_late_rate',
    'customer_late_rate'
]
# Create the preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Calculate scale_pos_weight for imbalanced classes
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Create the full pipeline
# We use default parameters for the model here, as the grid search will test new ones
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        random_state=42,
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss'
    ))
])

In [ ]:
# Define a (small) grid of parameters to search.
# A larger grid will be more accurate but will take much longer.
# Note: 'classifier__' prefix is crucial to tell the pipeline *which* step to apply the parameters to.
param_grid = {
    'classifier__max_depth': [4, 6],
    'classifier__n_estimators': [150, 250],
    'classifier__learning_rate': [0.1, 0.2]
}

# --- For a faster, but less thorough search, you can use RandomizedSearchCV ---
# from sklearn.model_selection import RandomizedSearchCV
# param_grid_random = {
#     'classifier__max_depth': [3, 4, 5, 6, 7],
#     'classifier__n_estimators': [100, 200, 300],
#     'classifier__learning_rate': [0.05, 0.1, 0.2],
#     'classifier__subsample': [0.7, 0.8, 0.9],
#     'classifier__colsample_bytree': [0.7, 0.8, 0.9]
# }
# grid_search = RandomizedSearchCV(pipeline, param_grid_random, n_iter=10, cv=3, scoring='accuracy', n_jobs=1, verbose=2, random_state=42)

In [7]:

# cv=3 means 3-fold cross-validation. Total fits = 2*2*2 * 3 = 24 models.
print("Starting GridSearchCV... This may take 30+ minutes.")
grid_search = GridSearchCV(
    pipeline, 
    param_grid, 
    cv=3, 
    scoring='f1_weighted',  # <-- CHANGED
    n_jobs=1,
    verbose=2
)

grid_search.fit(X_train, y_train)

print("GridSearchCV complete.")

Starting GridSearchCV... This may take 30+ minutes.
Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END classifier__learning_rate=0.1, classifier__max_depth=4, classifier__n_estimators=150; total time=   2.5s
[CV] END classifier__learning_rate=0.1, classifier__max_depth=4, classifier__n_estimators=150; total time=   0.7s
[CV] END classifier__learning_rate=0.1, classifier__max_depth=4, classifier__n_estimators=150; total time=   0.6s
[CV] END classifier__learning_rate=0.1, classifier__max_depth=4, classifier__n_estimators=250; total time=   0.8s
[CV] END classifier__learning_rate=0.1, classifier__max_depth=4, classifier__n_estimators=250; total time=   0.9s
[CV] END classifier__learning_rate=0.1, classifier__max_depth=4, classifier__n_estimators=250; total time=   0.9s
[CV] END classifier__learning_rate=0.1, classifier__max_depth=6, classifier__n_estimators=150; total time=   0.8s
[CV] END classifier__learning_rate=0.1, classifier__max_depth=6, classifier__n_estimators=

In [8]:
print(f"Best parameters found: {grid_search.best_params_}")
print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")

Best parameters found: {'classifier__learning_rate': 0.2, 'classifier__max_depth': 6, 'classifier__n_estimators': 250}
Best cross-validation accuracy: 0.7864


In [9]:
# Get the best model found by the search
best_model = grid_search.best_estimator_

# Evaluate it on the held-out test set
y_pred = best_model.predict(X_test)

print("\n--- Test Set Evaluation of Best Model ---")
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Set Accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


--- Test Set Evaluation of Best Model ---
Test Set Accuracy: 79.09%

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.85      0.79     16308
           1       0.86      0.74      0.80     19796

    accuracy                           0.79     36104
   macro avg       0.79      0.80      0.79     36104
weighted avg       0.80      0.79      0.79     36104



In [10]:
# Save the best *entire pipeline* (preprocessor + tuned model)
joblib.dump(best_model, '../models/delivery_risk_pipeline.joblib')
print("\nBest model pipeline saved to 'models/delivery_risk_pipeline.joblib'")


Best model pipeline saved to 'models/delivery_risk_pipeline.joblib'
